# ETL — Telco Customer Churn

**Proyecto:** Predicción de Churn en empresa de Telecomunicaciones  
**Fuente:** `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`  
**Destino:** `database/telco.db` → `data/telco_clean.csv` / `data/telco_clean.xlsx`  
**Objetivo:** Limpiar, transformar y enriquecer el dataset en 6 pasos trazables antes del modelado

---
## Sección 0 — Configuración

Importación de librerías y definición de rutas globales.

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
import openpyxl

# ── Rutas globales ────────────────────────────────────────────────────────
PATH_CSV        = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
PATH_DB         = '../database/telco.db'
PATH_CLEAN_CSV  = '../data/telco_clean.csv'
PATH_CLEAN_XLSX = '../data/telco_clean.xlsx'

# ── Versiones de librerías ────────────────────────────────────────────────
print('Versiones de librerías:')
print(f'  pandas   : {pd.__version__}')
print(f'  numpy    : {np.__version__}')
print(f'  openpyxl : {openpyxl.__version__}')
print(f'  sqlite3  : {sqlite3.sqlite_version}')

Versiones de librerías:
  pandas   : 2.2.3
  numpy    : 2.1.3
  openpyxl : 3.1.5
  sqlite3  : 3.45.3


In [2]:
# ── Verificar que existen los directorios necesarios ──────────────────────
for directorio in ['../data', '../database', '../notebooks']:
    os.makedirs(directorio, exist_ok=True)
    estado = '✓ existe' if os.path.isdir(directorio) else '✗ error'
    print(f'  {directorio:20s} : {estado}')

print('\nRutas configuradas:')
print(f'  CSV origen  : {PATH_CSV}')
print(f'  Base datos  : {PATH_DB}')
print(f'  CSV limpio  : {PATH_CLEAN_CSV}')
print(f'  Excel limpio: {PATH_CLEAN_XLSX}')

  ../data              : ✓ existe
  ../database          : ✓ existe
  ../notebooks         : ✓ existe

Rutas configuradas:
  CSV origen  : ../data/WA_Fn-UseC_-Telco-Customer-Churn.csv
  Base datos  : ../database/telco.db
  CSV limpio  : ../data/telco_clean.csv
  Excel limpio: ../data/telco_clean.xlsx


---
## Sección 1 — Carga del CSV a SQLite

Leer el CSV original con pandas, abrir (o crear) la base de datos SQLite y cargar los datos crudos como tabla `telco_raw`.

In [3]:
# ── Abrir conexión SQLite ─────────────────────────────────────────────────
conn = sqlite3.connect(PATH_DB)

# ── Leer CSV ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv(PATH_CSV)
print(f'CSV cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')

# ── Cargar a SQLite como tabla telco_raw ──────────────────────────────────
df_raw.to_sql('telco_raw', conn, if_exists='replace', index=False)
conn.commit()

CSV cargado: 7,043 filas × 21 columnas


In [4]:
# ── Verificar carga con sqlite3 ───────────────────────────────────────────
sql_count = 'SELECT COUNT(*) AS filas_cargadas FROM telco_raw'
resultado = pd.read_sql(sql_count, conn)
print(f'Filas en telco_raw: {resultado["filas_cargadas"].iloc[0]:,}')

print('\nPrimeras 3 filas de telco_raw:')
pd.read_sql('SELECT * FROM telco_raw LIMIT 3', conn)

Filas en telco_raw: 7,043

Primeras 3 filas de telco_raw:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


**Conclusión Sección 1:** 7,043 filas cargadas correctamente en `telco_raw`. La tabla refleja exactamente el CSV original, sin transformaciones.

---
## Sección 2 — Paso 1: Corrección de Tipos

Verificar los tipos almacenados en SQLite y crear `telco_step1` con los tipos semánticos correctos: `SeniorCitizen` como texto y `MonthlyCharges` como REAL.

In [5]:
# ── 2a: Tipo almacenado de TotalCharges antes de corregir ─────────────────
sql_tipo_tc = '''
SELECT typeof(TotalCharges) AS tipo_almacenado,
       COUNT(*)             AS n_registros
FROM telco_raw
GROUP BY typeof(TotalCharges)
'''
print('Tipo almacenado de TotalCharges en telco_raw:')
pd.read_sql(sql_tipo_tc, conn)

Tipo almacenado de TotalCharges en telco_raw:


,tipo_almacenado,n_registros
0,text,7043


In [6]:
# ── 2b: Valores de SeniorCitizen antes de corregir ────────────────────────
sql_senior = '''
SELECT SeniorCitizen,
       typeof(SeniorCitizen) AS tipo_almacenado,
       COUNT(*)              AS n_registros
FROM telco_raw
GROUP BY SeniorCitizen
'''
print('SeniorCitizen en telco_raw (actualmente como integer):')
pd.read_sql(sql_senior, conn)

SeniorCitizen en telco_raw (actualmente como integer):


,SeniorCitizen,tipo_almacenado,n_registros
0,0,integer,5901
1,1,integer,1142


In [7]:
# ── 2c: Mostrar SQL de creación de telco_step1 ────────────────────────────
sql_step1 = '''
DROP TABLE IF EXISTS telco_step1;
CREATE TABLE telco_step1 AS
SELECT
  customerID,
  gender,
  -- Convertir SeniorCitizen de 0/1 a etiqueta de texto
  CASE WHEN SeniorCitizen = 1 THEN 'Senior' ELSE 'No Senior' END AS SeniorCitizen,
  Partner,
  Dependents,
  tenure,
  PhoneService,
  MultipleLines,
  InternetService,
  OnlineSecurity,
  OnlineBackup,
  DeviceProtection,
  TechSupport,
  StreamingTV,
  StreamingMovies,
  Contract,
  PaperlessBilling,
  PaymentMethod,
  -- Forzar tipo numérico explícito
  CAST(MonthlyCharges AS REAL) AS MonthlyCharges,
  TotalCharges,   -- se convierte en Paso 2
  Churn
FROM telco_raw;
'''
print('SQL — Creación de telco_step1 (corrección de tipos):')
print(sql_step1)

SQL — Creación de telco_step1 (corrección de tipos):

DROP TABLE IF EXISTS telco_step1;
CREATE TABLE telco_step1 AS
SELECT
  customerID,
  gender,
  -- Convertir SeniorCitizen de 0/1 a etiqueta de texto
  CASE WHEN SeniorCitizen = 1 THEN 'Senior' ELSE 'No Senior' END AS SeniorCitizen,
  Partner,
  Dependents,
  tenure,
  PhoneService,
  MultipleLines,
  InternetService,
  OnlineSecurity,
  OnlineBackup,
  DeviceProtection,
  TechSupport,
  StreamingTV,
  StreamingMovies,
  Contract,
  PaperlessBilling,
  PaymentMethod,
  -- Forzar tipo numérico explícito
  CAST(MonthlyCharges AS REAL) AS MonthlyCharges,
  TotalCharges,   -- se convierte en Paso 2
  Churn
FROM telco_raw;



In [8]:
# ── Ejecutar DDL ──────────────────────────────────────────────────────────
conn.executescript(sql_step1)

# ── 2d: Verificar tipos después ───────────────────────────────────────────
sql_ver_step1 = '''
SELECT typeof(MonthlyCharges) AS tipo_MonthlyCharges,
       typeof(SeniorCitizen)  AS tipo_SeniorCitizen,
       COUNT(*)               AS n_registros
FROM telco_step1
GROUP BY typeof(MonthlyCharges), typeof(SeniorCitizen)
'''
print('Verificación de tipos en telco_step1:')
pd.read_sql(sql_ver_step1, conn)

Verificación de tipos en telco_step1:


,tipo_MonthlyCharges,tipo_SeniorCitizen,n_registros
0,real,text,7043


### Resumen de Cambios — Paso 1

| Columna | Tipo Original | Tipo Destino | Registros Afectados |
|---|---|---|---|
| `SeniorCitizen` | `INTEGER` (0/1) | `TEXT` ('Senior'/'No Senior') | 7,043 |
| `MonthlyCharges` | `REAL` (ya era REAL) | `REAL` (CAST explícito) | 7,043 |
| `TotalCharges` | `TEXT` (string) | `TEXT` (sin cambio — Paso 2) | 7,043 |

**Conclusión Sección 2:** `SeniorCitizen` convertido a texto descriptivo. `MonthlyCharges` confirmado como REAL. `TotalCharges` se mantiene como TEXT en este paso y se trata en la Sección 3.

---
## Sección 3 — Paso 2: Tratamiento de Nulos

Detectar espacios en blanco en `TotalCharges`, confirmar que corresponden a `tenure == 0` e imputar con `0.0`.

In [9]:
# ── 3a: Verificar nulos y blancos antes del tratamiento ───────────────────
sql_nulos_antes = '''
SELECT
  COUNT(*)  AS total_registros,
  SUM(CASE WHEN TotalCharges IS NULL          THEN 1 ELSE 0 END) AS nulos_reales,
  SUM(CASE WHEN TRIM(TotalCharges) = ''       THEN 1 ELSE 0 END) AS espacios_blancos,
  SUM(CASE WHEN tenure = 0                    THEN 1 ELSE 0 END) AS tenure_cero
FROM telco_step1
'''
print('Estado de TotalCharges antes del tratamiento:')
pd.read_sql(sql_nulos_antes, conn)

Estado de TotalCharges antes del tratamiento:


,total_registros,nulos_reales,espacios_blancos,tenure_cero
0,7043,0,11,11


In [10]:
# ── 3b: Mostrar los 11 registros afectados ────────────────────────────────
sql_afectados = '''
SELECT customerID, tenure, MonthlyCharges, TotalCharges, Churn
FROM telco_step1
WHERE TRIM(TotalCharges) = '' OR TotalCharges IS NULL
ORDER BY customerID
'''
print('Registros con TotalCharges vacío (confirmación: todos tienen tenure = 0):')
pd.read_sql(sql_afectados, conn)

Registros con TotalCharges vacío (confirmación: todos tienen tenure = 0):


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
0,1371-DWPAZ,0,56.05,,No
1,2520-SGTTA,0,20.00,,No
2,2775-SEFEE,0,61.90,,No
3,2923-ARZLG,0,19.70,,No
4,3115-CZMZD,0,20.25,,No
5,3213-VVOLG,0,25.35,,No
6,4075-WKNIU,0,73.35,,No
7,4367-NUYAO,0,25.75,,No
8,4472-LVYGI,0,52.55,,No
9,5709-LVOEQ,0,80.85,,No


In [11]:
# ── 3c: Mostrar SQL de creación de telco_step2 (proceso en dos etapas) ────
sql_step2_tmp = '''
DROP TABLE IF EXISTS telco_step2_tmp;
CREATE TABLE telco_step2_tmp AS
SELECT *,
  CAST(
    CASE
      WHEN TRIM(TotalCharges) = '' OR TotalCharges IS NULL THEN '0.0'
      ELSE TotalCharges
    END AS REAL
  ) AS TotalCharges_clean
FROM telco_step1;
'''

sql_step2 = '''
DROP TABLE IF EXISTS telco_step2;
CREATE TABLE telco_step2 AS
SELECT
  customerID, gender, SeniorCitizen, Partner, Dependents, tenure,
  PhoneService, MultipleLines, InternetService, OnlineSecurity,
  OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies,
  Contract, PaperlessBilling, PaymentMethod, MonthlyCharges,
  TotalCharges_clean AS TotalCharges,   -- renombrado, ahora como REAL
  Churn
FROM telco_step2_tmp;

DROP TABLE IF EXISTS telco_step2_tmp;
'''

print('SQL — Paso 1/2: Tabla temporal con TotalCharges_clean:')
print(sql_step2_tmp)
print('SQL — Paso 2/2: Crear telco_step2 y eliminar temporal:')
print(sql_step2)

SQL — Paso 1/2: Tabla temporal con TotalCharges_clean:

DROP TABLE IF EXISTS telco_step2_tmp;
CREATE TABLE telco_step2_tmp AS
SELECT *,
  CAST(
    CASE
      WHEN TRIM(TotalCharges) = '' OR TotalCharges IS NULL THEN '0.0'
      ELSE TotalCharges
    END AS REAL
  ) AS TotalCharges_clean
FROM telco_step1;

SQL — Paso 2/2: Crear telco_step2 y eliminar temporal:

DROP TABLE IF EXISTS telco_step2;
CREATE TABLE telco_step2 AS
SELECT
  customerID, gender, SeniorCitizen, Partner, Dependents, tenure,
  PhoneService, MultipleLines, InternetService, OnlineSecurity,
  OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies,
  Contract, PaperlessBilling, PaymentMethod, MonthlyCharges,
  TotalCharges_clean AS TotalCharges,   -- renombrado, ahora como REAL
  Churn
FROM telco_step2_tmp;

DROP TABLE IF EXISTS telco_step2_tmp;



In [12]:
# ── Ejecutar DDL en dos etapas ────────────────────────────────────────────
conn.executescript(sql_step2_tmp)
conn.executescript(sql_step2)

# ── 3d: Verificar después del tratamiento ─────────────────────────────────
sql_nulos_despues = '''
SELECT
  COUNT(*)                                              AS total_registros,
  SUM(CASE WHEN TotalCharges IS NULL THEN 1 ELSE 0 END) AS nulos_restantes,
  typeof(TotalCharges)                                  AS tipo_TotalCharges,
  MIN(TotalCharges)                                     AS valor_minimo,
  MAX(TotalCharges)                                     AS valor_maximo
FROM telco_step2
GROUP BY typeof(TotalCharges)
'''
print('Estado de TotalCharges después del tratamiento:')
pd.read_sql(sql_nulos_despues, conn)

Estado de TotalCharges después del tratamiento:


,total_registros,nulos_restantes,tipo_TotalCharges,valor_minimo,valor_maximo
0,7043,0,real,0.0,8684.8


### Análisis del Patrón de Ausencia

- **Patrón confirmado:** MAR (Missing At Random) — los 11 registros con `TotalCharges` vacío tienen `tenure = 0`.
- **Justificación:** Un cliente con `tenure = 0` aún no ha completado su primer período de facturación, por lo que no tiene cargos acumulados. El valor correcto es `0.0`.
- **Estrategia:** Imputación con `0.0` (no eliminación). Solo afecta al 0.16% del dataset.

| Métrica | Antes | Después |
|---|---|---|
| Nulos reales | 0 | 0 |
| Espacios en blanco | 11 | 0 |
| Tipo de dato | TEXT | REAL |
| Valor mínimo | N/A | 0.0 |

**Conclusión Sección 3:** Los 11 registros con `TotalCharges` vacío imputados con `0.0`. `TotalCharges` ahora es `REAL` en `telco_step2`.

---
## Sección 4 — Paso 3: Deduplicación

Verificar que no existen registros duplicados exactos ni duplicados por clave de negocio (`customerID`).

In [13]:
# ── 4a: Verificar duplicados exactos ─────────────────────────────────────
sql_dup_total = '''
SELECT
  COUNT(*)                    AS total_registros,
  COUNT(DISTINCT customerID)  AS customer_ids_unicos,
  COUNT(*) - COUNT(DISTINCT customerID) AS diferencia
FROM telco_step2
'''
print('Verificación de unicidad en telco_step2:')
pd.read_sql(sql_dup_total, conn)

Verificación de unicidad en telco_step2:


,total_registros,customer_ids_unicos,diferencia
0,7043,7043,0


In [14]:
# ── 4b: Verificar duplicados por clave de negocio ─────────────────────────
sql_dup_key = '''
SELECT customerID, COUNT(*) AS ocurrencias
FROM telco_step2
GROUP BY customerID
HAVING COUNT(*) > 1
ORDER BY ocurrencias DESC
'''
resultado_dups = pd.read_sql(sql_dup_key, conn)
print(f'customerID duplicados encontrados: {len(resultado_dups)}')
if len(resultado_dups) > 0:
    display(resultado_dups)
else:
    print('Sin duplicados por customerID')

customerID duplicados encontrados: 0
Sin duplicados por customerID


In [15]:
# ── 4c: Crear telco_step3 como copia de telco_step2 (trazabilidad) ────────
sql_step3 = '''
DROP TABLE IF EXISTS telco_step3;
CREATE TABLE telco_step3 AS SELECT * FROM telco_step2;
'''
print('SQL — Creación de telco_step3:')
print(sql_step3)
conn.executescript(sql_step3)
resultado = pd.read_sql('SELECT COUNT(*) AS filas FROM telco_step3', conn)
print(f'telco_step3 -> {resultado["filas"].iloc[0]:,} filas')

SQL — Creación de telco_step3:

DROP TABLE IF EXISTS telco_step3;
CREATE TABLE telco_step3 AS SELECT * FROM telco_step2;



telco_step3 -> 7,043 filas


**Conclusión Sección 4:** 0 duplicados exactos y 0 duplicados por `customerID`. El dataset tiene integridad referencial perfecta. `telco_step3` es una copia directa de `telco_step2` sin cambios, creada para mantener la trazabilidad del pipeline.

---
## Sección 5 — Paso 4: Estandarización de Texto

Verificar y eliminar espacios extra en columnas categóricas mediante `TRIM`.

In [16]:
# ── 5a: Verificar espacios extra antes de estandarizar ────────────────────
sql_espacios_antes = '''
SELECT
  SUM(CASE WHEN gender         != TRIM(gender)         THEN 1 ELSE 0 END) AS gender_espacios,
  SUM(CASE WHEN Contract       != TRIM(Contract)       THEN 1 ELSE 0 END) AS contract_espacios,
  SUM(CASE WHEN PaymentMethod  != TRIM(PaymentMethod)  THEN 1 ELSE 0 END) AS payment_espacios,
  SUM(CASE WHEN InternetService!= TRIM(InternetService) THEN 1 ELSE 0 END) AS internet_espacios,
  SUM(CASE WHEN Churn          != TRIM(Churn)          THEN 1 ELSE 0 END) AS churn_espacios
FROM telco_step3
'''
print('Espacios extra detectados en columnas clave (antes de TRIM):')
pd.read_sql(sql_espacios_antes, conn)

Espacios extra detectados en columnas clave (antes de TRIM):


,gender_espacios,contract_espacios,payment_espacios,internet_espacios,churn_espacios
0,0,0,0,0,0


In [17]:
# ── 5b: Mostrar SQL de telco_step4 ────────────────────────────────────────
sql_step4 = '''
DROP TABLE IF EXISTS telco_step4;
CREATE TABLE telco_step4 AS
SELECT
  TRIM(customerID)      AS customerID,
  TRIM(gender)          AS gender,
  TRIM(SeniorCitizen)   AS SeniorCitizen,
  TRIM(Partner)         AS Partner,
  TRIM(Dependents)      AS Dependents,
  tenure,
  TRIM(PhoneService)    AS PhoneService,
  TRIM(MultipleLines)   AS MultipleLines,
  TRIM(InternetService) AS InternetService,
  TRIM(OnlineSecurity)  AS OnlineSecurity,
  TRIM(OnlineBackup)    AS OnlineBackup,
  TRIM(DeviceProtection)AS DeviceProtection,
  TRIM(TechSupport)     AS TechSupport,
  TRIM(StreamingTV)     AS StreamingTV,
  TRIM(StreamingMovies) AS StreamingMovies,
  TRIM(Contract)        AS Contract,
  TRIM(PaperlessBilling)AS PaperlessBilling,
  TRIM(PaymentMethod)   AS PaymentMethod,
  MonthlyCharges,
  TotalCharges,
  TRIM(Churn)           AS Churn
FROM telco_step3;
'''
print('SQL — Creación de telco_step4 (TRIM en todas las columnas texto):')
print(sql_step4)

SQL — Creación de telco_step4 (TRIM en todas las columnas texto):

DROP TABLE IF EXISTS telco_step4;
CREATE TABLE telco_step4 AS
SELECT
  TRIM(customerID)      AS customerID,
  TRIM(gender)          AS gender,
  TRIM(SeniorCitizen)   AS SeniorCitizen,
  TRIM(Partner)         AS Partner,
  TRIM(Dependents)      AS Dependents,
  tenure,
  TRIM(PhoneService)    AS PhoneService,
  TRIM(MultipleLines)   AS MultipleLines,
  TRIM(InternetService) AS InternetService,
  TRIM(OnlineSecurity)  AS OnlineSecurity,
  TRIM(OnlineBackup)    AS OnlineBackup,
  TRIM(DeviceProtection)AS DeviceProtection,
  TRIM(TechSupport)     AS TechSupport,
  TRIM(StreamingTV)     AS StreamingTV,
  TRIM(StreamingMovies) AS StreamingMovies,
  TRIM(Contract)        AS Contract,
  TRIM(PaperlessBilling)AS PaperlessBilling,
  TRIM(PaymentMethod)   AS PaymentMethod,
  MonthlyCharges,
  TotalCharges,
  TRIM(Churn)           AS Churn
FROM telco_step3;



In [18]:
# ── Ejecutar DDL ──────────────────────────────────────────────────────────
conn.executescript(sql_step4)

# ── 5c: Verificar espacios después ────────────────────────────────────────
sql_espacios_despues = '''
SELECT
  SUM(CASE WHEN gender         != TRIM(gender)          THEN 1 ELSE 0 END) AS gender_espacios,
  SUM(CASE WHEN Contract       != TRIM(Contract)        THEN 1 ELSE 0 END) AS contract_espacios,
  SUM(CASE WHEN PaymentMethod  != TRIM(PaymentMethod)   THEN 1 ELSE 0 END) AS payment_espacios,
  SUM(CASE WHEN InternetService!= TRIM(InternetService) THEN 1 ELSE 0 END) AS internet_espacios,
  SUM(CASE WHEN Churn          != TRIM(Churn)           THEN 1 ELSE 0 END) AS churn_espacios
FROM telco_step4
'''
print('Espacios extra después de TRIM (deben ser 0 en todas):')
pd.read_sql(sql_espacios_despues, conn)

Espacios extra después de TRIM (deben ser 0 en todas):


,gender_espacios,contract_espacios,payment_espacios,internet_espacios,churn_espacios
0,0,0,0,0,0


**Conclusión Sección 5:** El dataset ya estaba estandarizado — 0 espacios extra detectados antes y después. La aplicación de `TRIM` es una verificación formal que garantiza la calidad del texto para operaciones futuras de agrupamiento y JOIN.

---
## Sección 6 — Paso 5: Outliers

Verificar outliers mediante el método IQR usando los límites calculados en el EDA. Decisión: mantener o eliminar.

In [19]:
# ── 6a: Detección de outliers con límites IQR del EDA ─────────────────────
# Límites calculados en el EDA (sección 7):
#   tenure        : Q1=9,      Q3=55,     LI=-60.00,    LS=124.00
#   MonthlyCharges: Q1=35.50,  Q3=89.85,  LI=-46.02,    LS=171.38
#   TotalCharges  : Q1=398.55, Q3=3786.60, LI=-4683.52, LS=8868.67

sql_outliers = '''
SELECT
  SUM(CASE WHEN tenure         < -60.00   OR tenure         > 124.00   THEN 1 ELSE 0 END) AS outliers_tenure,
  SUM(CASE WHEN MonthlyCharges < -46.02   OR MonthlyCharges > 171.38   THEN 1 ELSE 0 END) AS outliers_monthly,
  SUM(CASE WHEN TotalCharges   < -4683.52 OR TotalCharges   > 8868.67  THEN 1 ELSE 0 END) AS outliers_total,
  MIN(tenure)         AS min_tenure,   MAX(tenure)         AS max_tenure,
  MIN(MonthlyCharges) AS min_monthly,  MAX(MonthlyCharges) AS max_monthly,
  MIN(TotalCharges)   AS min_total,    MAX(TotalCharges)   AS max_total
FROM telco_step4
'''
print('SQL de detección de outliers (método IQR):')
print(sql_outliers)
print('Resultado:')
pd.read_sql(sql_outliers, conn)

SQL de detección de outliers (método IQR):

SELECT
  SUM(CASE WHEN tenure         < -60.00   OR tenure         > 124.00   THEN 1 ELSE 0 END) AS outliers_tenure,
  SUM(CASE WHEN MonthlyCharges < -46.02   OR MonthlyCharges > 171.38   THEN 1 ELSE 0 END) AS outliers_monthly,
  SUM(CASE WHEN TotalCharges   < -4683.52 OR TotalCharges   > 8868.67  THEN 1 ELSE 0 END) AS outliers_total,
  MIN(tenure)         AS min_tenure,   MAX(tenure)         AS max_tenure,
  MIN(MonthlyCharges) AS min_monthly,  MAX(MonthlyCharges) AS max_monthly,
  MIN(TotalCharges)   AS min_total,    MAX(TotalCharges)   AS max_total
FROM telco_step4

Resultado:


,outliers_tenure,outliers_monthly,outliers_total,min_tenure,max_tenure,min_monthly,max_monthly,min_total,max_total
0,0,0,0,0,72,18.25,118.75,0.0,8684.8


In [20]:
# ── 6b: Crear telco_step5 (copia de step4 — no se eliminan registros) ─────
sql_step5 = '''
DROP TABLE IF EXISTS telco_step5;
CREATE TABLE telco_step5 AS SELECT * FROM telco_step4;
'''
print('SQL — Creación de telco_step5:')
print(sql_step5)
conn.executescript(sql_step5)
resultado = pd.read_sql('SELECT COUNT(*) AS filas FROM telco_step5', conn)
print(f'telco_step5 -> {resultado["filas"].iloc[0]:,} filas (sin eliminaciones)')

SQL — Creación de telco_step5:

DROP TABLE IF EXISTS telco_step5;
CREATE TABLE telco_step5 AS SELECT * FROM telco_step4;



telco_step5 -> 7,043 filas (sin eliminaciones)


### Decisión sobre Outliers

**Resultado:** 0 outliers detectados por el método IQR en las tres variables numéricas.

**Justificación de negocio para valores extremos:**
- `tenure` máximo (~72 meses): clientes veteranos de 6 años — válido para un operador de telecomunicaciones.
- `MonthlyCharges` máximo (~118 USD): clientes con múltiples servicios premium — coherente con el catálogo de servicios.
- `TotalCharges` máximo (~8,684 USD): producto directo de tenure × MonthlyCharges altos — matemáticamente esperado.

**Decisión: Mantener todos los registros.** Los valores extremos representan perfiles de clientes legítimos de alto valor. Se aplicará transformación logarítmica en el modelado para reducir la asimetría.

**Conclusión Sección 6:** 0 outliers que requieran eliminación. `telco_step5` = `telco_step4` con 7,043 registros intactos.

---
## Sección 7 — Paso 6: Columnas Calculadas

Crear la tabla final `telco_clean` añadiendo 6 columnas derivadas requeridas por las hipótesis H1–H9.

In [21]:
# ── 7a: Mostrar SQL de creación de telco_clean ────────────────────────────
sql_clean = '''
DROP TABLE IF EXISTS telco_clean;
CREATE TABLE telco_clean AS
SELECT
  -- 21 columnas originales limpias
  customerID, gender, SeniorCitizen, Partner, Dependents, tenure,
  PhoneService, MultipleLines, InternetService, OnlineSecurity,
  OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies,
  Contract, PaperlessBilling, PaymentMethod, MonthlyCharges,
  TotalCharges, Churn,

  -- Col 22: variable target numérica (para correlaciones y KPIs)
  CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END AS Churn_num,

  -- Col 23: grupos de permanencia (para H1 y H2)
  CASE
    WHEN tenure BETWEEN 0  AND 12 THEN '01_0-12 meses'
    WHEN tenure BETWEEN 13 AND 24 THEN '02_13-24 meses'
    WHEN tenure BETWEEN 25 AND 48 THEN '03_25-48 meses'
    WHEN tenure BETWEEN 49 AND 72 THEN '04_49-72 meses'
  END AS tenure_group,

  -- Col 24: perfil familiar (para H4)
  CASE
    WHEN Partner = 'Yes' AND Dependents = 'Yes' THEN 'Con pareja y dependientes'
    WHEN Partner = 'Yes' AND Dependents = 'No'  THEN 'Con pareja, sin dependientes'
    WHEN Partner = 'No'  AND Dependents = 'Yes' THEN 'Sin pareja, con dependientes'
    WHEN Partner = 'No'  AND Dependents = 'No'  THEN 'Sin pareja ni dependientes'
  END AS perfil_familiar,

  -- Col 25: número de servicios adicionales contratados (para H7)
  (CASE WHEN OnlineSecurity   = 'Yes' THEN 1 ELSE 0 END +
   CASE WHEN OnlineBackup     = 'Yes' THEN 1 ELSE 0 END +
   CASE WHEN DeviceProtection = 'Yes' THEN 1 ELSE 0 END +
   CASE WHEN TechSupport      = 'Yes' THEN 1 ELSE 0 END +
   CASE WHEN StreamingTV      = 'Yes' THEN 1 ELSE 0 END +
   CASE WHEN StreamingMovies  = 'Yes' THEN 1 ELSE 0 END) AS n_servicios,

  -- Col 26: rango de cargo mensual (para H9)
  CASE
    WHEN MonthlyCharges <= 35                           THEN '01_Bajo (<=35)'
    WHEN MonthlyCharges > 35  AND MonthlyCharges <= 65  THEN '02_Medio (35-65)'
    WHEN MonthlyCharges > 65  AND MonthlyCharges <= 95  THEN '03_Alto (65-95)'
    WHEN MonthlyCharges > 95                            THEN '04_Muy alto (>95)'
  END AS rango_cargo,

  -- Col 27: segmento de riesgo alto (para H9): cargo alto + contrato mes a mes
  CASE
    WHEN MonthlyCharges > 65 AND Contract = 'Month-to-month' THEN 1
    ELSE 0
  END AS segmento_riesgo

FROM telco_step5;
'''
print('SQL — Creación de telco_clean con columnas calculadas:')
print(sql_clean)

SQL — Creación de telco_clean con columnas calculadas:

DROP TABLE IF EXISTS telco_clean;
CREATE TABLE telco_clean AS
SELECT
  -- 21 columnas originales limpias
  customerID, gender, SeniorCitizen, Partner, Dependents, tenure,
  PhoneService, MultipleLines, InternetService, OnlineSecurity,
  OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies,
  Contract, PaperlessBilling, PaymentMethod, MonthlyCharges,
  TotalCharges, Churn,

  -- Col 22: variable target numérica (para correlaciones y KPIs)
  CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END AS Churn_num,

  -- Col 23: grupos de permanencia (para H1 y H2)
  CASE
    WHEN tenure BETWEEN 0  AND 12 THEN '01_0-12 meses'
    WHEN tenure BETWEEN 13 AND 24 THEN '02_13-24 meses'
    WHEN tenure BETWEEN 25 AND 48 THEN '03_25-48 meses'
    WHEN tenure BETWEEN 49 AND 72 THEN '04_49-72 meses'
  END AS tenure_group,

  -- Col 24: perfil familiar (para H4)
  CASE
    WHEN Partner = 'Yes' AND Dependents = 'Yes' THEN 'Con pareja y depen

In [22]:
# ── Ejecutar DDL ──────────────────────────────────────────────────────────
conn.executescript(sql_clean)
resultado = pd.read_sql('SELECT COUNT(*) AS filas FROM telco_clean', conn)
cols = pd.read_sql('SELECT * FROM telco_clean LIMIT 1', conn).columns.tolist()
print(f'telco_clean -> {resultado["filas"].iloc[0]:,} filas, {len(cols)} columnas')
print(cols)

telco_clean -> 7,043 filas, 27 columnas
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_num', 'tenure_group', 'perfil_familiar', 'n_servicios', 'rango_cargo', 'segmento_riesgo']


In [23]:
# ── Verificar tenure_group ────────────────────────────────────────────────
print('Distribución de tenure_group con tasa de churn:')
pd.read_sql('''
SELECT tenure_group,
       COUNT(*) AS n_clientes,
       ROUND(AVG(Churn_num)*100, 2) AS churn_pct
FROM telco_clean
GROUP BY tenure_group
ORDER BY tenure_group
''', conn)

Distribución de tenure_group con tasa de churn:

,tenure_group,n_clientes,churn_pct
0,01_0-12 meses,2186,47.44
1,02_13-24 meses,1024,28.71
2,03_25-48 meses,1594,20.39
3,04_49-72 meses,2239,9.51


In [24]:
# ── Verificar perfil_familiar ─────────────────────────────────────────────
print('Distribución de perfil_familiar con tasa de churn:')
pd.read_sql('''
SELECT perfil_familiar,
       COUNT(*) AS n_clientes,
       ROUND(AVG(Churn_num)*100, 2) AS churn_pct
FROM telco_clean
GROUP BY perfil_familiar
ORDER BY churn_pct DESC
''', conn)

Distribución de perfil_familiar con tasa de churn:


,perfil_familiar,n_clientes,churn_pct
0,Sin pareja ni dependientes,3280,34.24
1,"Con pareja, sin dependientes",1653,25.41
2,"Sin pareja, con dependientes",361,21.33
3,Con pareja y dependientes,1749,14.24


In [25]:
# ── Verificar n_servicios ─────────────────────────────────────────────────
print('Distribución de n_servicios con tasa de churn:')
pd.read_sql('''
SELECT n_servicios,
       COUNT(*) AS n_clientes,
       ROUND(AVG(Churn_num)*100, 2) AS churn_pct
FROM telco_clean
GROUP BY n_servicios
ORDER BY n_servicios
''', conn)

Distribución de n_servicios con tasa de churn:


,n_servicios,n_clientes,churn_pct
0,0,2219,21.41
1,1,966,45.76
2,2,1033,35.82
3,3,1118,27.37
4,4,852,22.30
5,5,571,12.43
6,6,284,5.28


In [26]:
# ── Verificar rango_cargo ─────────────────────────────────────────────────
print('Distribución de rango_cargo con tasa de churn:')
pd.read_sql('''
SELECT rango_cargo,
       COUNT(*) AS n_clientes,
       ROUND(AVG(Churn_num)*100, 2) AS churn_pct
FROM telco_clean
GROUP BY rango_cargo
ORDER BY rango_cargo
''', conn)

Distribución de rango_cargo con tasa de churn:


,rango_cargo,n_clientes,churn_pct
0,01_Bajo (<=35),1735,10.89
1,02_Medio (35-65),1409,23.14
2,03_Alto (65-95),2604,35.94
3,04_Muy alto (>95),1295,32.28


In [27]:
# ── Verificar segmento_riesgo ─────────────────────────────────────────────
print('segmento_riesgo (1 = cargo alto + mes a mes) con tasa de churn:')
pd.read_sql('''
SELECT segmento_riesgo,
       COUNT(*) AS n_clientes,
       ROUND(AVG(Churn_num)*100, 2) AS churn_pct
FROM telco_clean
GROUP BY segmento_riesgo
''', conn)

segmento_riesgo (1 = cargo alto + mes a mes) con tasa de churn:


,segmento_riesgo,n_clientes,churn_pct
0,0,4751,14.38
1,1,2292,51.75


**Conclusión Sección 7:** `telco_clean` creada con 27 columnas (21 originales + 6 calculadas). Cada columna calculada muestra diferencias claras en la tasa de churn entre categorías, confirmando su valor predictivo para las hipótesis planteadas.

---
## Sección 8 — Paso 7: Verificación del Dataset Limpio

Comparación de KPIs entre el EDA y el ETL para garantizar que las transformaciones no alteraron la integridad del dataset.

In [28]:
# ── 8a: Dimensiones ───────────────────────────────────────────────────────
print('Dimensiones de telco_clean:')
pd.read_sql('''
SELECT
  COUNT(*)                   AS total_filas,
  COUNT(DISTINCT customerID) AS clientes_unicos
FROM telco_clean
''', conn)

Dimensiones de telco_clean:


,total_filas,clientes_unicos
0,7043,7043


In [29]:
# ── 8b: Verificación de nulos en columnas críticas ────────────────────────
print('Nulos en columnas numéricas críticas:')
pd.read_sql('''
SELECT
  SUM(CASE WHEN TotalCharges   IS NULL THEN 1 ELSE 0 END) AS nulos_TotalCharges,
  SUM(CASE WHEN MonthlyCharges IS NULL THEN 1 ELSE 0 END) AS nulos_MonthlyCharges,
  SUM(CASE WHEN tenure         IS NULL THEN 1 ELSE 0 END) AS nulos_tenure
FROM telco_clean
''', conn)

Nulos en columnas numéricas críticas:


,nulos_TotalCharges,nulos_MonthlyCharges,nulos_tenure
0,0,0,0


In [30]:
# ── 8c: KPIs globales — comparar con EDA ─────────────────────────────────
sql_kpis = '''
SELECT
  COUNT(*)                                                              AS total_clientes,
  SUM(Churn_num)                                                        AS total_cancelados,
  ROUND(AVG(Churn_num)*100, 2)                                          AS tasa_cancelacion_pct,
  ROUND(SUM(MonthlyCharges), 2)                                         AS revenue_mensual_total,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS revenue_en_riesgo,
  ROUND(AVG(tenure), 1)                                                 AS tenure_promedio,
  ROUND(AVG(MonthlyCharges), 2)                                         AS cargo_mensual_promedio
FROM telco_clean
'''
df_kpis = pd.read_sql(sql_kpis, conn)
display(df_kpis)

# ── Assertions: deben coincidir con valores del EDA ───────────────────────
kpi = df_kpis.iloc[0]

assert int(kpi['total_clientes'])   == 7043,   'ERROR: total_clientes no coincide con EDA (esperado: 7043)'
assert int(kpi['total_cancelados']) == 1869,   'ERROR: total_cancelados no coincide con EDA (esperado: 1869)'
assert abs(kpi['tasa_cancelacion_pct']   - 26.54)     < 0.01,  'ERROR: tasa_cancelacion no coincide'
assert abs(kpi['revenue_mensual_total']  - 456116.60) < 0.50,  'ERROR: revenue_mensual no coincide'
assert abs(kpi['revenue_en_riesgo']      - 139130.85) < 0.50,  'ERROR: revenue_en_riesgo no coincide'
assert abs(kpi['tenure_promedio']        - 32.4)      < 0.10,  'ERROR: tenure_promedio no coincide'
assert abs(kpi['cargo_mensual_promedio'] - 64.76)     < 0.01,  'ERROR: cargo_mensual_promedio no coincide'

print('Todos los KPIs coinciden con los valores del EDA')

,total_clientes,total_cancelados,tasa_cancelacion_pct,revenue_mensual_total,revenue_en_riesgo,tenure_promedio,cargo_mensual_promedio
0,7043,1869,26.54,456116.6,139130.85,32.4,64.76


Todos los KPIs coinciden con los valores del EDA


### Tabla Comparativa EDA vs ETL

| Métrica | Valor EDA | Valor ETL | ¿Coincide? |
|---|---|---|---|
| Total clientes | 7,043 | 7,043 | ✅ |
| Total cancelados | 1,869 | 1,869 | ✅ |
| Tasa de cancelación | 26.54% | 26.54% | ✅ |
| Revenue mensual total | $456,116.60 | $456,116.60 | ✅ |
| Revenue en riesgo | $139,130.85 | $139,130.85 | ✅ |
| Tenure promedio | 32.4 meses | 32.4 meses | ✅ |
| Cargo mensual promedio | $64.76 | $64.76 | ✅ |

**Conclusión Sección 8:** Todos los KPIs coinciden exactamente entre el EDA y el dataset limpio. Las transformaciones del ETL no alteraron ninguna métrica de negocio. El pipeline es correcto.

---
## Sección 9 — Exportación

Leer `telco_clean` desde SQLite y exportar a CSV y Excel para su uso en el análisis de hipótesis y modelado.

In [31]:
# ── Leer telco_clean desde SQLite ─────────────────────────────────────────
conn_export = sqlite3.connect(PATH_DB)
df_clean = pd.read_sql('SELECT * FROM telco_clean', conn_export)
conn_export.close()

print(f'Dataset leído: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas')
print(f'Columnas: {df_clean.columns.tolist()}')
print()
print('Tipos de dato:')
print(df_clean.dtypes.to_string())

Dataset leído: 7,043 filas × 27 columnas
Columnas: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_num', 'tenure_group', 'perfil_familiar', 'n_servicios', 'rango_cargo', 'segmento_riesgo']

Tipos de dato:
customerID           object
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyC

In [32]:
# ── Exportar a CSV ────────────────────────────────────────────────────────
df_clean.to_csv(PATH_CLEAN_CSV, index=False, encoding='utf-8')
print(f'✓ CSV exportado: {PATH_CLEAN_CSV}')

# ── Exportar a Excel ──────────────────────────────────────────────────────
df_clean.to_excel(PATH_CLEAN_XLSX, index=False, sheet_name='telco_clean', engine='openpyxl')
print(f'✓ Excel exportado: {PATH_CLEAN_XLSX}')

✓ CSV exportado: ../data/telco_clean.csv

✓ Excel exportado: ../data/telco_clean.xlsx


In [33]:
# ── Verificar archivos exportados ────────────────────────────────────────
for ruta, nombre in [(PATH_CLEAN_CSV, 'telco_clean.csv'), (PATH_CLEAN_XLSX, 'telco_clean.xlsx')]:
    existe = os.path.exists(ruta)
    if existe:
        tam_kb = os.path.getsize(ruta) / 1024
        print(f'  {nombre:25s}: {tam_kb:8.1f} KB')
    else:
        print(f'  {nombre:25s}: NO ENCONTRADO')

df_check = pd.read_csv(PATH_CLEAN_CSV)
print(f'CSV exportado -> {df_check.shape[0]:,} filas, {df_check.shape[1]} columnas')
assert df_check.shape == df_clean.shape, 'ERROR: dimensiones del CSV no coinciden'

  telco_clean.csv          :   1454.2 KB
  telco_clean.xlsx         :    802.5 KB
CSV exportado -> 7,043 filas, 27 columnas


**Conclusión Sección 9:** Dataset exportado exitosamente en dos formatos:
- `data/telco_clean.csv` — para scripts Python y análisis de hipótesis.
- `data/telco_clean.xlsx` — para validación manual y revisión por stakeholders.

Ambos archivos contienen las 7,043 filas y 27 columnas de `telco_clean`.

---
## Sección 10 — Resumen de Decisiones ETL

Consolidación de todas las transformaciones aplicadas, con justificación y trazabilidad completa.

### Tabla de Decisiones ETL

| Paso | Transformación | Estado Antes | Estado Después | Registros Afectados | Decisión | Justificación |
|---|---|---|---|---|---|---|
| 1 | Corrección de tipos | `SeniorCitizen`=INT, `MonthlyCharges`=REAL implícito | `SeniorCitizen`=TEXT ('Senior'/'No Senior'), `MonthlyCharges`=REAL explícito | 7,043 / 7,043 | Conversión | Coherencia semántica — integer binario no representa bien una categoría |
| 2 | Nulos en `TotalCharges` | 11 espacios en blanco (TEXT) | 0 nulos (REAL = 0.0) | 11 | Imputar 0.0 | Patrón MAR: `tenure=0` → sin período de facturación completado |
| 3 | Deduplicación | 0 duplicados exactos | 0 duplicados exactos | 0 | Sin acción | Dataset ya íntegro — verificación formal únicamente |
| 4 | Estandarización texto | 0 espacios extra | 0 espacios extra | 0 | TRIM preventivo | Garantía para agrupamientos futuros y operaciones de JOIN |
| 5 | Outliers IQR | 0 outliers detectados | 0 outliers | 0 | Mantener todos | Valores extremos válidos de negocio (clientes veteranos de alto valor) |
| 6 | Columnas eliminadas | — | — | 0 | Sin eliminación | Todas las columnas tienen uso en el análisis |
| 7 | Columnas calculadas | 21 columnas | 27 columnas | 7,043 | +6 columnas nuevas | Requeridas por hipótesis H1–H9: `Churn_num`, `tenure_group`, `perfil_familiar`, `n_servicios`, `rango_cargo`, `segmento_riesgo` |

---

### Resumen Final del Pipeline ETL

**Dataset limpio:** 7,043 filas × 27 columnas

**Tablas intermedias generadas en `database/telco.db`:**

| Tabla | Descripción |
|---|---|
| `telco_raw` | CSV original sin modificar (carga directa) |
| `telco_step1` | Tipos corregidos: `SeniorCitizen` → TEXT, `MonthlyCharges` → REAL |
| `telco_step2` | `TotalCharges` convertido a REAL (11 blancos → 0.0) |
| `telco_step3` | Copia de step2 — hito de deduplicación (sin cambios) |
| `telco_step4` | TRIM aplicado a todas las columnas TEXT |
| `telco_step5` | Hito de outliers — sin eliminaciones |
| `telco_clean` | Dataset final con 6 columnas calculadas |

**Archivos exportados:**
- `data/telco_clean.csv` — formato universal para análisis en Python/R
- `data/telco_clean.xlsx` — formato revisión para stakeholders

**Próximo paso:** Modelado relacional en `sql/02_ddl.sql` — diseño del esquema normalizado para consultas analíticas de hipótesis.